# K-Means: Segmentacion de Contenido

Agrupa los 15,000 titulos en clusters segun sus caracteristicas de calidad y rendimiento.
No hay target: es aprendizaje no supervisado.

Features: `imdb_rating`, `rotten_tomatoes_score`, `votos_log`, `presupuesto_log`, `awards_won`, `hours_watched_million`, `duracion_total`, `antiguedad`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score


## 1. Cargar datos y escalar

In [ ]:
df = pd.read_csv('../data/processed/catalogo_procesado.csv')
FEATURES_KM = [
    'imdb_rating', 'rotten_tomatoes_score', 'votos_log', 'presupuesto_log',
    'awards_won', 'hours_watched_million', 'duracion_total', 'antiguedad',
]

df_km = df[FEATURES_KM].dropna().copy()
print(f'Filas para clustering: {len(df_km)}')

scaler_km = StandardScaler()
X_km = scaler_km.fit_transform(df_km)


## 2. Determinar k optimo

Tres criterios complementarios:
- **Inercia (Elbow)**: buscar el codo donde la mejora decrece
- **Silhouette**: mayor es mejor (clusters mas compactos y separados)
- **Davies-Bouldin**: menor es mejor

In [ ]:
inercias, silhouettes, davies_bouldins = [], [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_km)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_km, labels, sample_size=3000, random_state=42))
    davies_bouldins.append(davies_bouldin_score(X_km, labels))
    print(f'k={k}: inercia={km.inertia_:.0f}, silhouette={silhouettes[-1]:.3f}, DB={davies_bouldins[-1]:.3f}')


In [ ]:
k_vals = list(K_range)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(k_vals, inercias, 'o-', color='steelblue')
axes[0].set_title('Inercia (Elbow)'); axes[0].set_xlabel('k')

axes[1].plot(k_vals, silhouettes, 'o-', color='darkorange')
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k')

axes[2].plot(k_vals, davies_bouldins, 'o-', color='seagreen')
axes[2].set_title('Davies-Bouldin Index'); axes[2].set_xlabel('k')

plt.suptitle('Criterios para seleccion de k', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/18_criterios_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Modelo final con k=4

Se elige k=4 como equilibrio entre interpretabilidad y calidad del clustering (silhouette local max).

In [ ]:
K_OPTIMO = 4
kmeans = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df_km['cluster'] = kmeans.fit_predict(X_km)

print('Distribucion de clusters:')
print(df_km['cluster'].value_counts().sort_index())


## 4. Perfiles de clusters

Se compara cada cluster con la media global para identificar sus caracteristicas distintivas.

Perfiles identificados (segun las medias observadas):
- **Cluster 0 (Blockbuster Costoso):** mega-hits con millones de horas vistas y muchos votos IMDb
- **Cluster 1 (Nicho Subestimado):** sin presupuesto reportado, audiencia pequena, notas moderadas
- **Cluster 2 (Joya de Critica):** IMDb y RT altos, audiencia moderada, alto reconocimiento
- **Cluster 3 (Contenido Promedio):** presupuesto medio, notas bajas, audiencia tipica

In [ ]:
medias = df_km.groupby('cluster')[FEATURES_KM].mean()
media_global = df_km[FEATURES_KM].mean()

print('Medias por cluster:')
print(medias.round(2).to_string())


In [ ]:
std_global = df_km[FEATURES_KM].std()
medias_centradas = (medias - media_global) / std_global

nombres_clusters = {
    0: 'Blockbuster Costoso',
    1: 'Nicho Subestimado',
    2: 'Joya de Critica',
    3: 'Contenido Promedio',
}

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(medias_centradas.values, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
ax.set_xticks(range(len(FEATURES_KM)))
ax.set_xticklabels(FEATURES_KM, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(K_OPTIMO))
ax.set_yticklabels([f'C{i}: {nombres_clusters[i]}' for i in range(K_OPTIMO)])
plt.colorbar(im, ax=ax, label='Desviaciones de la media global')
ax.set_title('Perfil de clusters (desviacion respecto a media global)')
plt.tight_layout()
plt.savefig('../reports/figures/19_heatmap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Proyeccion PCA-3D

In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_km)
varianza = pca.explained_variance_ratio_
colores_c = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
for i in range(K_OPTIMO):
    mask = df_km['cluster'].values == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
               c=colores_c[i], label=f'C{i}: {nombres_clusters[i]}',
               alpha=0.5, s=10)
ax.set_xlabel(f'PC1 ({varianza[0]:.1%})')
ax.set_ylabel(f'PC2 ({varianza[1]:.1%})')
ax.set_zlabel(f'PC3 ({varianza[2]:.1%})')
ax.set_title('Clusters en espacio PCA-3D')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/figures/20_pca3d_clusters.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Guardar modelo y etiquetas

In [ ]:
joblib.dump(kmeans, '../models/kmeans.joblib')

cluster_labels_df = df[FEATURES_KM].dropna().copy()
cluster_labels_df['cluster'] = df_km['cluster'].values
cluster_labels_df['cluster_nombre'] = cluster_labels_df['cluster'].map(nombres_clusters)
cluster_labels_df.to_csv('../data/processed/cluster_labels.csv', index=True)

print('Guardado: models/kmeans.joblib')
print('Guardado: data/processed/cluster_labels.csv')
print(f'Total titulos con cluster asignado: {len(cluster_labels_df)}')
